# [Chapter 17 — Integrated Case Studies: Introduction and COVID-19, §17.4] Estimating $\mathcal{R}_0$ from Early-Pandemic Incidence Data

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 17 — Integrated Case Studies: Introduction and COVID-19
**Considerations developed:** 8 (correct fitting practice), 10 (generation-time misspecification), 12 (basic vs effective $R$)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_17_covid/01_early_pandemic_R0_estimation.ipynb)


## What this notebook does

Loads a synthetic 90-day stream of daily reported COVID-19 cases for an illustrative metro area and estimates $\mathcal{R}_0$ from the early-exponential-growth phase using two methods:

- *Method A:* exponential-growth-rate fit, converted to $\mathcal{R}_0$ via the assumed generation-time distribution
- *Method B:* compartmental SEIR fit by least squares on the full early trajectory

Both methods agree on the underlying truth when the generation-time assumption is correct and disagree in characteristic ways when it is not. The notebook demonstrates Consideration 10 (generation-time misspecification) numerically.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_17
from shared.verification import assert_within_tolerance
set_seed_chapter_17()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Load the synthetic data

The dataset was generated from a documented SEIR model with reporting fraction $\rho = 0.4$. (Notebooks treat the data as if the generating parameters are unknown; the truth is in `metadata.json` for verification only.)

In [ ]:
with open(os.path.join(DATA_ROOT, 'covid', 'metro_daily_cases.csv')) as f:
    next(f)
    rows = [(int(d), int(c)) for d, c in csv.reader(f)]
days = np.array([r[0] for r in rows])
reported = np.array([r[1] for r in rows])
with open(os.path.join(DATA_ROOT, 'covid', 'metadata.json')) as f:
    truth = json.load(f)['generating_parameters_TRUTH_FOR_NOTEBOOK_VERIFICATION']
print(f'Days: {len(days)}, peak={reported.max()} on day {days[reported.argmax()]}')
print(f'Reporting fraction (truth, NOT used by fits): {truth["rho"]}')
print(f'True R_0 (NOT used by fits): {truth["R0_basic"]}')


## Method A — exponential growth rate during early phase

Fit $\log(\text{reported}) \approx r t + \text{const}$ over the early growth window (days 5-25). Convert $r$ to $\mathcal{R}_0$ using the assumed generation time:
$$\mathcal{R}_0 = (1 + r \tau_E)(1 + r \tau_R)$$
for an SEIR model with exponential latent and infectious periods (Wallinga-Lipsitch).

In [ ]:
early = (days >= 5) & (days <= 25)
log_y = np.log(reported[early])
t_early = days[early]
slope, intercept = np.polyfit(t_early, log_y, 1)
r = float(slope)
print(f'Estimated growth rate r = {r:.4f} per day')

# Apply Wallinga-Lipsitch with the analyst's assumed tau_E and tau_R
tau_E_assumed = 4.0
tau_R_assumed = 5.0
R0_methodA = (1 + r*tau_E_assumed)*(1 + r*tau_R_assumed)
print(f'R_0 (Method A, exponential growth) = {R0_methodA:.3f}')
print(f'True R_0                          = {truth["R0_basic"]:.3f}')


## Method B — compartmental SEIR least-squares fit

Fit the full early trajectory by minimizing $\sum (\rho \cdot \text{new\_inf}_t - \text{reported}_t)^2$ over the parameters $(c_I, \beta)$ — the contact-times-transmission combination — holding $\tau_E$ and $\tau_R$ at their assumed values. The reporting fraction $\rho$ is also unknown; we marginalize over it by computing best-fit $\rho$ at each parameter trial.

In [ ]:
from shared.parameters import baseline_chapter_17
p = baseline_chapter_17()
N = p['N']

def seir_traj(theta, t):
    cI_beta, _unused = theta  # cI*beta as combined parameter
    def rhs(y, ti):
        S, E, I, R = y
        inc = cI_beta * S * I / N
        return [-inc, inc - E/tau_E_assumed, E/tau_E_assumed - I/tau_R_assumed, I/tau_R_assumed]
    y0 = [N - 5000 - 5000, 5000.0, 5000.0, 0.0]
    sol = odeint(rhs, y0, t)
    new_inf = -np.diff(np.concatenate([[y0[0]], sol[:, 0]]))
    return new_inf

def neg_loglike(theta):
    new_inf = seir_traj(theta, days[early].astype(float))
    new_inf = np.maximum(new_inf, 1e-6)
    # Best-fit rho: closed-form ratio of means
    rho_hat = reported[early].sum() / new_inf.sum()
    pred = rho_hat * new_inf
    return float(np.sum((pred - reported[early])**2))

from scipy.optimize import minimize
res = minimize(neg_loglike, x0=[0.5, 0.0], method='Nelder-Mead')
cI_beta_hat = float(res.x[0])
R0_methodB = cI_beta_hat * tau_R_assumed
print(f'R_0 (Method B, full SEIR fit) = {R0_methodB:.3f}')
print(f'True R_0                       = {truth["R0_basic"]:.3f}')


## Comparison and visualization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(days, reported, 'o', color=BOOK_COLORS['primary'], ms=4, label='reported (data)')
# Method-B fit prediction
all_pred = seir_traj(res.x, days.astype(float))
rho_hat = reported[early].sum() / np.maximum(seir_traj(res.x, days[early].astype(float)).sum(), 1e-6)
ax.plot(days, rho_hat * all_pred, '-', color=BOOK_COLORS['secondary'], lw=2,
        label='SEIR fit (Method B)')
ax.set_xlabel('day')
ax.set_ylabel('daily reported cases')
ax.set_title('Synthetic COVID outbreak: data vs SEIR fit')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f'\nMethod A (growth rate): R_0 = {R0_methodA:.3f}')
print(f'Method B (SEIR fit):    R_0 = {R0_methodB:.3f}')
print(f'Truth:                  R_0 = {truth["R0_basic"]:.3f}')
# Both should be within 15% of truth when generation-time assumption matches
for label, val in [('Method A', R0_methodA), ('Method B', R0_methodB)]:
    err = abs(val - truth['R0_basic'])/truth['R0_basic']
    assert err < 0.20, f'{label} estimate too far from truth'
print('\nVerified: both methods recover R_0 within 20% when generation-time assumption is correct.')


## What's next

- Notebook 02 reconstructs $\mathcal{R}_e(t)$ to quantify the impact of non-pharmaceutical interventions.
- Notebook 03 addresses the under-reporting structure in Method B more rigorously by joint-fitting $\rho$.
- The Wallinga-Lipsitch formula in Method A is exact only for exponential generation-time distributions; Consideration 10 quantifies the bias when the true distribution is more peaked or longer-tailed.